In [ ]:
import numpy as np
import pandas as pd
import glob
import gc

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import RFE, VarianceThreshold

from skopt import BayesSearchCV  
from sklearn.metrics import roc_auc_score, f1_score, recall_score, accuracy_score

from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.feature_selection import RFECV
import matplotlib.pyplot as plt

In [ ]:
dat0 = pd.read_csv('traindat.csv')
data_x=dat0.drop(['Label'],axis=1)
y=np.array(dat0['Label'])

cat_cols = data_x.select_dtypes(include=['object']).columns

mean_cols = ['BMI']
median_object_cols = ['Education', 'Pre.Residence', 'Pre.Work']
median_num_cols = ['Economic.conditions', 'Pre.Income', 'Pre.Drink']

data_x_imp = data_x.copy()
imputer_mean = SimpleImputer(strategy="mean")
data_x_imp[mean_cols] = imputer_mean.fit_transform(data_x_imp[mean_cols])

imputer_median_obj = SimpleImputer(strategy="most_frequent")
data_x_imp[median_object_cols] = imputer_median_obj.fit_transform(data_x_imp[median_object_cols])

imputer_median_num = SimpleImputer(strategy="median")
data_x_imp[median_num_cols] = imputer_median_num.fit_transform(data_x_imp[median_num_cols])

# --- dummy ---
X_all_dum = pd.get_dummies(data_x_imp, columns=cat_cols, dtype=int)
all_features = X_all_dum.columns.tolist()
print(f"总特征数: {len(all_features)}")

y_all=y

In [ ]:
#############################################################
##### RFECV and RFE(K=20)
# random_seed=721;1123;666

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)

results = {
    'model': [],         
    'fold': [],           
    'rfecv_k': [],          # only for RFECV-based
    'selected_features': [] 
}

def preprocess_features(X_train, X_val):
    vt = VarianceThreshold(threshold=0.01)
    X_train = vt.fit_transform(X_train)
    X_val = vt.transform(X_val)
    
    return X_train, X_val, vt.get_support()
    
def get_rfe_estimator(model_name):
    if model_name == 'rf':
        return RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=2
        )
    elif model_name == 'xgb':
        return xgb.XGBClassifier(
            n_estimators=50,
            max_depth=4,
            reg_alpha=0.1,
            reg_lambda=0.01,
            random_state=42
        )
    elif model_name == 'lgb':
        return lgb.LGBMClassifier(
            n_estimators=50,
            max_depth=4,
            num_leaves=15,
            reg_lambda=0.01,
            random_state=42,
            verbose=-1
        )

In [ ]:
########## RFECV-based
for model_name in ['rf', 'xgb', 'lgb']:
    for fold, (train_idx, val_idx) in enumerate(outer_cv.split(data_x, y), 1):
        X_train_raw = data_x.iloc[train_idx].copy()
        X_val_raw = data_x.iloc[val_idx].copy()
        y_train = y[train_idx]
        y_val = y[val_idx]

        imputer_mean = SimpleImputer(strategy="mean")
        X_train_raw[mean_cols] = imputer_mean.fit_transform(X_train_raw[mean_cols])
        
        imputer_median_obj = SimpleImputer(strategy="most_frequent")
        X_train_raw[median_object_cols] = imputer_median_obj.fit_transform(X_train_raw[median_object_cols])
        
        imputer_median_num = SimpleImputer(strategy="median")
        X_train_raw[median_num_cols] = imputer_median_num.fit_transform(X_train_raw[median_num_cols])
        
        X_train_dum = pd.get_dummies(X_train_raw, columns=cat_cols, dtype=int)
        
        for col in all_features:
            if col not in X_train_dum.columns:
                X_train_dum[col] = 0
        X_train_dum = X_train_dum[all_features]
    
        X_val_raw[mean_cols] = imputer_mean.transform(X_val_raw[mean_cols])
        X_val_raw[median_object_cols] = imputer_median_obj.transform(X_val_raw[median_object_cols])
        X_val_raw[median_num_cols] = imputer_median_num.transform(X_val_raw[median_num_cols])
        
        X_val_dum = pd.get_dummies(X_val_raw, columns=cat_cols, dtype=int)
        for col in all_features:
            if col not in X_val_dum.columns:
                X_val_dum[col] = 0
        X_val_dum = X_val_dum[all_features]

        X_train = X_train_dum
        X_val = X_val_dum
        
        X_train_pre, X_val_pre, vt_support = preprocess_features(X_train, X_val)

        pre_features = [X_train.columns.tolist()[i] for i in np.where(vt_support)[0]]

        rfe_estimator = get_rfe_estimator(model_name)

        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

        rfecv = RFECV(
            estimator=rfe_estimator,
            step=1,
            cv=inner_cv,
            scoring='roc_auc',
            min_features_to_select=1,
            n_jobs=2
        )
        rfecv.fit(X_train_pre, y_train)

        optimal_k = rfecv.n_features_
        
        selected_feats = [pre_features[i] for i in np.where(rfecv.support_)[0]]

        results['model'].append(model_name)
        results['fold'].append(fold)
        results['rfecv_k'].append(optimal_k)
        results['selected_features'].append(selected_feats)

results_df = pd.DataFrame(results)
results_df.to_csv(f'cv{random_seed}_RFECVresults_detail.csv', index=False)

In [ ]:
########## RFE (K=20)
for model_name in ['rf', 'xgb', 'lgb']:
    for fold, (train_idx, val_idx) in enumerate(outer_cv.split(data_x, y), 1):
        X_train_raw = data_x.iloc[train_idx].copy()
        X_val_raw = data_x.iloc[val_idx].copy()
        y_train = y[train_idx]
        y_val = y[val_idx]

        imputer_mean = SimpleImputer(strategy="mean")
        X_train_raw[mean_cols] = imputer_mean.fit_transform(X_train_raw[mean_cols])
        
        imputer_median_obj = SimpleImputer(strategy="most_frequent")
        X_train_raw[median_object_cols] = imputer_median_obj.fit_transform(X_train_raw[median_object_cols])
        
        imputer_median_num = SimpleImputer(strategy="median")
        X_train_raw[median_num_cols] = imputer_median_num.fit_transform(X_train_raw[median_num_cols])
        
        X_train_dum = pd.get_dummies(X_train_raw, columns=cat_cols, dtype=int)
        
        for col in all_features:
            if col not in X_train_dum.columns:
                X_train_dum[col] = 0
        X_train_dum = X_train_dum[all_features]
    
        X_val_raw[mean_cols] = imputer_mean.transform(X_val_raw[mean_cols])
        X_val_raw[median_object_cols] = imputer_median_obj.transform(X_val_raw[median_object_cols])
        X_val_raw[median_num_cols] = imputer_median_num.transform(X_val_raw[median_num_cols])
        
        X_val_dum = pd.get_dummies(X_val_raw, columns=cat_cols, dtype=int)
        for col in all_features:
            if col not in X_val_dum.columns:
                X_val_dum[col] = 0
        X_val_dum = X_val_dum[all_features]

        X_train = X_train_dum
        X_val = X_val_dum
        
        X_train_pre, X_val_pre, vt_support = preprocess_features(X_train, X_val)

        pre_features = [X_train.columns.tolist()[i] for i in np.where(vt_support)[0]]

        rfe_estimator = get_rfe_estimator(model_name)

        optimal_k=20
        rfe = RFE(estimator=rfe_estimator, 
                  n_features_to_select=optimal_k, 
                  step=1)

        selected_feats = [pre_features[i] for i in np.where(rfe.support_)[0]]

        results['model'].append(model_name)
        results['fold'].append(fold)
        results['selected_features'].append(selected_feats)

results_df = pd.DataFrame(results)
results_df.to_csv(f'cv{random_seed}_RFEresults_detail.csv', index=False)

In [ ]:
########## Feature frequency
# folder = 'RFE';'RFECV'
detail_files = glob.glob(os.path.join(folder, '*results_detail.csv'))

all_feature_lists = []
for file in detail_files:
    df = pd.read_csv(file)
    for feat_str in df['selected_features']:
        feat_list = feat_str.strip('[]').replace("'", "").split(', ')
        feat_list = [f.strip() for f in feat_list if f.strip()]
        all_feature_lists.append(feat_list)

feature_counter = Counter()
for feat_list in all_feature_lists:
    for feat in feat_list:
        feature_counter[feat] += 1

total_lists = len(all_feature_lists)
feature_freq = {feat: count/total_lists for feat, count in feature_counter.items()}

sorted_features = sorted(feature_freq.items(), key=lambda x: x[1], reverse=True)
print(f"\nUnique Features: {len(sorted_features)}")
print("Top 20 features:")
for feat, freq in sorted_features[:20]:
    print(f"  {feat}: {freq:.2%}")

df_sorted = pd.DataFrame(sorted_features, columns=['feature', 'frequency'])
df_sorted.to_csv('sorted_features.csv', index=False)

sorted_feat_names = df_sorted['feature']

In [ ]:
model_names = ['rf', 'xgb', 'lgb']
random_seed = 123

k_values = list(range(1, 51, 1))

results_summary = []

for k in k_values:
    
    selected_features = sorted_feat_names[:k]
    
    for model_name in model_names:
        train_aucs = []
        val_aucs = []
        outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
        
        for fold, (train_idx, val_idx) in enumerate(outer_cv.split(data_x, y), 1):
            X_train_raw = data_x.iloc[train_idx].copy()
            X_val_raw = data_x.iloc[val_idx].copy()
            y_train_fold = y[train_idx]
            y_val_fold = y[val_idx]
            
            imputer_mean = SimpleImputer(strategy="mean")
            X_train_raw[mean_cols] = imputer_mean.fit_transform(X_train_raw[mean_cols])
            
            imputer_median_obj = SimpleImputer(strategy="most_frequent")
            X_train_raw[median_object_cols] = imputer_median_obj.fit_transform(X_train_raw[median_object_cols])
            
            imputer_median_num = SimpleImputer(strategy="median")
            X_train_raw[median_num_cols] = imputer_median_num.fit_transform(X_train_raw[median_num_cols])
            
            X_train_dum = pd.get_dummies(X_train_raw, columns=cat_cols, dtype=int)
            
            for col in all_features:
                if col not in X_train_dum.columns:
                    X_train_dum[col] = 0
            X_train_dum = X_train_dum[all_features]
        
            X_val_raw[mean_cols] = imputer_mean.transform(X_val_raw[mean_cols])
            X_val_raw[median_object_cols] = imputer_median_obj.transform(X_val_raw[median_object_cols])
            X_val_raw[median_num_cols] = imputer_median_num.transform(X_val_raw[median_num_cols])
            
            X_val_dum = pd.get_dummies(X_val_raw, columns=cat_cols, dtype=int)
            for col in all_features:
                if col not in X_val_dum.columns:
                    X_val_dum[col] = 0
            X_val_dum = X_val_dum[all_features]
    
            X_train_fold = X_train_dum[selected_features]
            X_val_fold = X_val_dum[selected_features]
            
            model= get_rfe_estimator(model_name)
            model.fit(X_train_fold, y_train_fold)
            
            train_pred_fold = model.predict_proba(X_train_fold)[:, 1]
            val_pred_fold = model.predict_proba(X_val_fold)[:, 1]
            
            train_auc_fold = roc_auc_score(y_train_fold, train_pred_fold)
            val_auc_fold = roc_auc_score(y_val_fold, val_pred_fold)
            
            train_aucs.append(train_auc_fold)
            val_aucs.append(val_auc_fold)
        
        mean_train_auc = np.mean(train_aucs)
        std_train_auc = np.std(train_aucs)
        mean_val_auc = np.mean(val_aucs)
        std_val_auc = np.std(val_aucs)
    
        results_summary.append({
            'k': k,
            'model': model_name,
            'mean_train_auc': mean_train_auc,
            'std_train_auc': std_train_auc,
            'mean_validation_auc': mean_val_auc,
            'std_validation_auc': std_val_auc,
            'n_features': len(selected_features)
        })
        
        print(f"{model_name} K={k}: 5折平均AUC={mean_val_auc:.4f} (±{std_val_auc:.4f})")

        df_results = pd.DataFrame(results_summary)
        df_results.to_csv(f'{folder}feature_count_vs_auc_rfeparm.csv', index=False)

In [ ]:
###################################################
##### LASSO and EN
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.compose import make_column_selector
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegressionCV

In [ ]:
# Preprocessor
class CustomPreprocessor3(BaseEstimator, TransformerMixin):
    def __init__(self, mean_cols, median_object_cols, median_num_cols,
                 cat_cols, all_features):
        self.mean_cols = mean_cols
        self.median_object_cols = median_object_cols
        self.median_num_cols = median_num_cols
        self.cat_cols = cat_cols
        self.all_features = all_features
        self.num_cols = [c for c in all_features if c not in cat_cols]

    def fit(self, X, y=None):
        # imputers
        self.imputer_mean_ = SimpleImputer(strategy="mean")
        self.imputer_mean_.fit(X[self.mean_cols])

        self.imputer_median_obj_ = SimpleImputer(strategy="most_frequent")
        self.imputer_median_obj_.fit(X[self.median_object_cols])

        self.imputer_median_num_ = SimpleImputer(strategy="median")
        self.imputer_median_num_.fit(X[self.median_num_cols])

        # dummy + align
        X_tmp = self._impute_only(X)
        X_dum = pd.get_dummies(X_tmp, columns=self.cat_cols, dtype=int)

        for col in self.all_features:
            if col not in X_dum.columns:
                X_dum[col] = 0
        X_dum = X_dum[self.all_features]

        # scaler（仅数值列）
        self.num_cols_after_dummy_ = [
            c for c in self.num_cols if c in X_dum.columns
        ]
        self.scaler_ = StandardScaler()
        self.scaler_.fit(X_dum[self.num_cols_after_dummy_])

        self.feature_names_out_ = X_dum.columns
        return self

    def transform(self, X):
        X = X.copy()
        X = self._impute_only(X)

        X_dum = pd.get_dummies(X, columns=self.cat_cols, dtype=int)
        for col in self.all_features:
            if col not in X_dum.columns:
                X_dum[col] = 0
        X_dum = X_dum[self.all_features]

        X_dum[self.num_cols_after_dummy_] = self.scaler_.transform(
            X_dum[self.num_cols_after_dummy_]
        )
        return X_dum

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

    def _impute_only(self, X):
        X = X.copy()
        X[self.mean_cols] = self.imputer_mean_.transform(X[self.mean_cols])
        X[self.median_object_cols] = self.imputer_median_obj_.transform(
            X[self.median_object_cols]
        )
        X[self.median_num_cols] = self.imputer_median_num_.transform(
            X[self.median_num_cols]
        )
        return X

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, selected_features):
        self.selected_features = selected_features

    def fit(self, X, y=None):
        self.feature_names_ = X.columns   # ✅ 只有 DataFrame 才有 .columns
        return self

    def transform(self, X):
        return X[self.selected_features]  # ✅ 返回 DataFrame

    def get_feature_names_out(self, input_features=None):
        return np.array(self.selected_features)

In [ ]:
########### Logit LASSO 第一轮筛选

random_seed=123

# pipeline
pipeline_logit_lasso = Pipeline([
    ('preprocessor', CustomPreprocessor3(
        mean_cols=mean_cols,
        median_object_cols=median_object_cols,
        median_num_cols=median_num_cols,
        cat_cols=cat_cols,
        all_features=all_features 
    )),
    ('model', LogisticRegressionCV(
        penalty='l1',
        solver='liblinear',
        Cs=np.logspace(-2, 2, 50), 
        cv=5,
        scoring='roc_auc',
        max_iter=10000,
        random_state=random_seed
    ))
])

pipeline_logit_lasso.fit(data_x, y)

feature_names_logit_lasso = pipeline_logit_lasso.named_steps['preprocessor'].get_feature_names_out()

coef_logit_lasso = pipeline_logit_lasso.named_steps['model'].coef_[0]  # 注意这里是 [0]，因为二分类

selected_mask = np.abs(coef_logit_lasso) > 1e-8
selected_features_logit_lasso = feature_names_logit_lasso[selected_mask].tolist()
selected_coefs_logit_lasso = coef_logit_lasso[selected_mask]

results_logit_lasso = pd.DataFrame({
    'feature_name': selected_features_logit_lasso,
    'coefficient': selected_coefs_logit_lasso,
    'abs_coefficient': np.abs(selected_coefs_logit_lasso)
}).sort_values('abs_coefficient', ascending=False)

results_logit_lasso.to_csv('logistic_lasso_selected_features.csv', index=False)

In [ ]:
########### Logit EN 第一轮
random_seed=123

pipeline_logit_en = Pipeline([
    ('preprocessor', CustomPreprocessor3(
        mean_cols=mean_cols,
        median_object_cols=median_object_cols,
        median_num_cols=median_num_cols,
        cat_cols=cat_cols,
        all_features=all_features   # ✅ 原始 all_features
    )),
    ('model', LogisticRegressionCV(
        penalty='elasticnet',
        solver='saga',
        l1_ratios=[0.3,0.5,0.7],
        Cs=np.logspace(-2, 2, 50),   # 30 个候选 C
        cv=5,
        scoring='roc_auc',
        max_iter=10000,
        random_state=random_seed
    ))
])

pipeline_logit_en.fit(data_x, y)

feature_names_logit_en = pipeline_logit_en.named_steps['preprocessor'].get_feature_names_out()

coef_logit_en = pipeline_logit_en.named_steps['model'].coef_[0]

selected_mask_en = np.abs(coef_logit_en) > 1e-8
selected_features_logit_en = feature_names_logit_en[selected_mask_en].tolist()
selected_coefs_logit_en = coef_logit_en[selected_mask_en]

results_logit_en = pd.DataFrame({
    'feature_name': selected_features_logit_en,
    'coefficient': selected_coefs_logit_en,
    'abs_coefficient': np.abs(selected_coefs_logit_en)
}).sort_values('abs_coefficient', ascending=False)

results_logit_en.to_csv('logistic_en_selected_features.csv', index=False)